# 1. Le JSON, sans réseau

In [1]:
import os
print(os.getcwd())

c:\Users\Julie\code\json-api-geocodage\notebooks


In [2]:
import os

print("je m'execute depuis :", os.getcwd())
print("le dossier data existe :", os.path.isdir("../data"))
print("le fichier existe     :", os.path.isfile("../data/restaurants.json"))
print("contenu de ../data    :", os.listdir("../data"))

je m'execute depuis : c:\Users\Julie\code\json-api-geocodage\notebooks
le dossier data existe : True
le fichier existe     : True
contenu de ../data    : ['restaurants.json', 'villes.json']


In [3]:
import json

with open("../data/restaurants.json", encoding="utf-8") as f:
    data = json.load(f)

print(type(data))
print(data["ville"], "-", data["pays"])
print("nombre de restaurants :", len(data["restaurants"]))

<class 'dict'>
Catane - Italie
nombre de restaurants : 4


In [4]:
print(data["restaurants"][0]["nom"])
print(data["restaurants"][0]["notes"][0])

Gelateria Zio Pietro dal 1964
4.6


In [5]:
for r in data["restaurants"]:
    moyenne = sum(r["notes"]) / len(r["notes"])
    print(f"{r['nom']:32} {moyenne:.2f}")

Gelateria Zio Pietro dal 1964    4.70
A Casa d'Amici                   4.45
La Bitta                         4.15
Steak House                      4.00


In [6]:
print(json.dumps(data["restaurants"][1], ensure_ascii=False, indent=2))

{
  "nom": "A Casa d'Amici",
  "adresse": "Via Fischetti, 14, 95131 Catania CT",
  "type": "trattoria",
  "notes": [
    4.4,
    4.5
  ]
}


# 2. Du JSON au DataFrame

## Cours

In [7]:
import pandas as pd

df = pd.json_normalize(data)
print(df.shape)
print(df.columns.tolist())

(1, 4)
['ville', 'pays', 'releve', 'restaurants']


In [8]:
df = pd.json_normalize(data, record_path="restaurants")
df

,nom,adresse,type,notes
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,"[4.6, 4.8, 4.7]"
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,"[4.4, 4.5]"
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,"[4.2, 4.0, 4.3, 4.1]"
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,"[3.9, 4.1]"


In [9]:
df = pd.json_normalize(data, record_path="restaurants", meta=["ville", "pays"])
df

,nom,adresse,type,notes,ville,pays
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,"[4.6, 4.8, 4.7]",Catane,Italie
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,"[4.4, 4.5]",Catane,Italie
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,"[4.2, 4.0, 4.3, 4.1]",Catane,Italie
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,"[3.9, 4.1]",Catane,Italie


## Challenge 1 — Un JSON à deux étages 

In [10]:
with open("../data/villes.json", encoding="utf-8") as f:
    villes_data = json.load(f)

In [11]:

villes_data

{'pays': 'Italie',
 'releve': '2026-11-05',
 'villes': [{'nom': 'Catane',
   'region': 'Sicile',
   'habitants': 298762,
   'restaurants': [{'nom': 'Gelateria Zio Pietro dal 1964',
     'type': 'glacier',
     'notes': [4.6, 4.8, 4.7]},
    {'nom': "A Casa d'Amici", 'type': 'trattoria', 'notes': [4.4, 4.5]},
    {'nom': 'La Bitta', 'type': 'poisson', 'notes': [4.2, 4.0, 4.3, 4.1]}]},
  {'nom': 'Palerme',
   'region': 'Sicile',
   'habitants': 630828,
   'restaurants': [{'nom': 'Antica Focacceria',
     'type': 'trattoria',
     'notes': [4.3, 4.1, 4.4]},
    {'nom': 'Ferro di Cavallo', 'type': 'trattoria', 'notes': [4.5, 4.6]}]},
  {'nom': 'Syracuse',
   'region': 'Sicile',
   'habitants': 121605,
   'restaurants': [{'nom': 'Don Camillo',
     'type': 'poisson',
     'notes': [4.7, 4.8, 4.6, 4.9]},
    {'nom': 'Gelateria Bianca', 'type': 'glacier', 'notes': [4.2]}]}]}

In [12]:
print("Nombre de villes :", len(villes_data["villes"]))
print("Nom de la deuxieme ville :", villes_data["villes"][1]["nom"])
print("Nombre de restaurants à Syracuse", len(villes_data["villes"][2]["restaurants"]))
print("Nom du premier restaurant de Palerme :", villes_data["villes"][1]["restaurants"][0]["nom"])

Nombre de villes : 3
Nom de la deuxieme ville : Palerme
Nombre de restaurants à Syracuse 2
Nom du premier restaurant de Palerme : Antica Focacceria


In [14]:
for ville in villes_data["villes"]:
    print(ville["nom"], ":" , len(ville["restaurants"]), "restaurants")

Catane : 3 restaurants
Palerme : 2 restaurants
Syracuse : 2 restaurants


In [15]:
df_villes = pd.json_normalize(
    villes_data,
    record_path=["villes", "restaurants"],
    meta=[
        ["villes", "nom"],
        ["villes", "region"],
        ["villes", "habitants"]
    ]
)

df_villes

,nom,type,notes,villes.nom,villes.region,villes.habitants
0,Gelateria Zio Pietro dal 1964,glacier,"[4.6, 4.8, 4.7]",Catane,Sicile,298762
1,A Casa d'Amici,trattoria,"[4.4, 4.5]",Catane,Sicile,298762
2,La Bitta,poisson,"[4.2, 4.0, 4.3, 4.1]",Catane,Sicile,298762
3,Antica Focacceria,trattoria,"[4.3, 4.1, 4.4]",Palerme,Sicile,630828
4,Ferro di Cavallo,trattoria,"[4.5, 4.6]",Palerme,Sicile,630828
5,Don Camillo,poisson,"[4.7, 4.8, 4.6, 4.9]",Syracuse,Sicile,121605
6,Gelateria Bianca,glacier,[4.2],Syracuse,Sicile,121605


In [18]:
moyennes = []

for notes in df_villes["notes"]:
    moyenne = sum(notes) / len(notes)
    moyennes.append(round(moyenne, 2))

df_villes["note_moyenne"] = moyennes

df_villes

,nom,type,notes,villes.nom,villes.region,villes.habitants,note_moyenne
0,Gelateria Zio Pietro dal 1964,glacier,"[4.6, 4.8, 4.7]",Catane,Sicile,298762,4.70
1,A Casa d'Amici,trattoria,"[4.4, 4.5]",Catane,Sicile,298762,4.45
2,La Bitta,poisson,"[4.2, 4.0, 4.3, 4.1]",Catane,Sicile,298762,4.15
3,Antica Focacceria,trattoria,"[4.3, 4.1, 4.4]",Palerme,Sicile,630828,4.27
4,Ferro di Cavallo,trattoria,"[4.5, 4.6]",Palerme,Sicile,630828,4.55
5,Don Camillo,poisson,"[4.7, 4.8, 4.6, 4.9]",Syracuse,Sicile,121605,4.75
6,Gelateria Bianca,glacier,[4.2],Syracuse,Sicile,121605,4.20


In [20]:
df_villes.sort_values(by="note_moyenne", ascending=False).head(1)

,nom,type,notes,villes.nom,villes.region,villes.habitants,note_moyenne
5,Don Camillo,poisson,"[4.7, 4.8, 4.6, 4.9]",Syracuse,Sicile,121605,4.75


In [21]:
df_villes.groupby("type").agg("nom").count().sort_values(ascending=False)

type
trattoria    3
glacier      2
poisson      2
Name: nom, dtype: int64

# 3. Une API, c'est un JSON au bout d'une URL

## Cours

In [22]:
import requests

url = "https://api-adresse.data.gouv.fr/search/"
r = requests.get(url, params={"q": "20 avenue de Ségur 75007 Paris", "limit": 1})

print(r.status_code)

200


In [23]:
reponse = r.json()
print(type(reponse))
print(reponse.keys())

<class 'dict'>
dict_keys(['type', 'features', 'query'])


In [24]:
r = requests.get(url, params={"q": "20 avenue de Ségur 75007 Paris", "limit": 1})
print(r.url)

https://api-adresse.data.gouv.fr/search/?q=20+avenue+de+S%C3%A9gur+75007+Paris&limit=1


In [25]:
headers = {"User-Agent": "bootcamp-simplon-da/1.0 (prenom.nom@exemple.fr)"}
r = requests.get(url, params={"q": "Catane"}, headers=headers)

print(r.status_code)
print(r.json()["features"][0]["properties"]["label"])

200
Catane 31310 Montesquieu-Volvestre


In [26]:
import requests

def interroger(url, params, headers=None):
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        return r.json()
    except requests.exceptions.Timeout:
        print("Trop lent, on abandonne cette requête")
    except requests.exceptions.HTTPError as e:
        print("Le serveur a répondu une erreur :", e)
    except requests.exceptions.RequestException as e:
        print("Problème réseau :", e)
    return None

In [27]:
print(interroger("https://api-adresse.data.gouv.fr/search/", {"q": "Lille", "limit": 1}) is not None)
print(interroger("https://api-adresse.data.gouv.fr/nimportequoi", {"q": "Lille"}))


True
Le serveur a répondu une erreur : 404 Client Error: Not Found for url: https://api-adresse.data.gouv.fr/nimportequoi?q=Lille
None


## Challenge 2 — Interroger une API en série

In [29]:
adresses = [
    "20 avenue de Ségur 75007 Paris",
    "Grand Place, Lille",
    "1 rue de la Paix, Paris",
    "place Bellecour, Lyon",
    "zzzzz qwerty 99999",
]

In [35]:
r = requests.get(
    url,
    params={"q": adresses[0], "limit": 1},
    timeout=10
)

print("Code HTTP :", r.status_code)

reponse = r.json()

print("Label :", reponse["features"][0]["properties"]["label"])
print("Score :", reponse["features"][0]["properties"]["score"])

Code HTTP : 200
Label : 20 Avenue de Ségur 75007 Paris
Score : 0.9716281818181818


In [36]:
for adresse in adresses:
    reponse = interroger(url, {"q": adresse, "limit": 1})
    if reponse is not None and len(reponse["features"]) > 0:
        print(adresse, "->", reponse["features"][0]["properties"]["label"])
    else:
        print(adresse, "-> pas de résultat")

20 avenue de Ségur 75007 Paris -> 20 Avenue de Ségur 75007 Paris
Grand Place, Lille -> Place de Gand 59800 Lille
1 rue de la Paix, Paris -> 1 Rue de la Paix 75002 Paris
place Bellecour, Lyon -> Place Bellecour 69002 Lyon
zzzzz qwerty 99999 -> pas de résultat
